In [4]:
##### Cleans Argentina labor data
# reformats

import os
import pandas as pd

In [5]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
labor = pd.ExcelFile(f"{cd}/Data/Raw/Sub_National/Argentina/labor.xlsx")

ARG_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/ARG_departments.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Labor/ARG_labor.csv"

In [6]:
### Pull together labor data from different tabs

# Get tab names with data
tabs = [s for s in labor.sheet_names if s not in ['Índice', '7.14']]

# gather all labor data 
dfs = []
for tab in tabs:
    df = pd.read_excel(
        labor,
        sheet_name=tab,
        header=None,
        skiprows=10,
        usecols='B,C,D'
    )
    df.columns = ['department', 'type', 'value']
    
    # Forward-fill department names into the blank Personas rows
    df['department'] = df['department'].ffill()
    
    # Keep only EAP and Personas rows, drop any footer/blank rows
    df = df[df['type'].isin(['EAP', 'Personas'])]
    
    # Pivot so each department is one row with columns for EAP and Personas
    df = df.pivot(index='department', columns='type', values='value').reset_index()
    df.columns.name = None
    dfs.append(df)

labor_merged = pd.concat(dfs, ignore_index=True)[['department', 'EAP', 'Personas']]

# convert ot number 
labor_merged["Personas"] = pd.to_numeric(
    labor_merged["Personas"].replace("-", 0),
    errors="coerce"
)

labor_merged = labor_merged.dropna()

/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_53652/3883935353.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labor_merged["Personas"].replace("-", 0),


In [8]:
##### Clean labor
# merge with region data
labor_merged = labor_merged.merge(ARG_codes, on='department', how='right')

# rename columns
labor_merged.rename(columns={'GID_2': 'GEO_ID'}, inplace=True)

labor_merged.rename(columns={'Personas': 'ag_labor_jobs'}, inplace=True)

# add info
labor_merged['ISO3'] = 'ARG'
labor_merged['Year'] = 2018
labor_merged['GEO_ID_NAME'] = 'GID_2'

# select columns
col_to_keep = ['ISO3', 'GEO_ID', 'GEO_ID_NAME', 'Year', 'ag_labor_jobs']
labor_merged = labor_merged[col_to_keep]
labor_merged['ag_labor_jobs'] = labor_merged['ag_labor_jobs'].fillna(0)

In [9]:
##### Save cleaned data
labor_merged.to_csv(save_path, index=False)